# Week 6 · Multithreading and Parallelism

**Programming for Data Science (606315)** · University of Petra

At the end of each semester the Registration Department sends every faculty a CSV file of marks. Loading them one
after another means each file waits for the previous one, even though the computer has several CPU cores sitting idle
and most of the time is spent waiting for the disk or the network. This week is about doing several things **at
once**: when that helps, when it does not, and how to do it safely in Python.

This notebook follows the lecture slides section by section. Their examples are used as they are, with explanation,
diagrams and timing experiments added; a few boxes mark where current Python differs from the slides (**Side note**)
or add a topic the slides do not cover (**Beyond the slides**). Threads, `concurrent.futures`, `Queue` and `Lock`
also come from the course's threading notebook.

| Section | Topic | Slides |
|---|---|---|
| 1 | What multiprocessing is, and why data scientists care | 3–4, 7 |
| 2 | Concurrency vs parallelism: threads and processes | 5 |
| 3 | The Global Interpreter Lock (GIL), measured | 6 |
| 4 | Before you start a process: start methods, pickling and the `__main__` guard | 7, 9, 15 |
| 5 | The `Process` class; separate memory and `Queue` | 8–9, 15 |
| 6 | Process pools: `apply()` vs `apply_async()` | 10–13 |
| 7 | The context manager, `map` and friends, errors in workers | 16–18 |
| 8 | `concurrent.futures`: thread and process pool executors | 20 |
| 9 | Threads that share data: `Lock` and `Queue` | 7, 15 |
| 10 | Practical: parallel data loading with Pandas | 14 |
| 11 | Decision guide, pitfalls, best practices, further reading | 15–21 |

> **How to run this notebook.** Run the cells in order. The worker functions that child processes run live in
> [`workers.py`](workers.py) next to this notebook (Section 4 explains why); open it in a second tab.

In [ ]:
import sys, time, threading, queue, multiprocessing
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from pathlib import Path

# find this week's folder (works from the repo root, the week folder, or solutions/)
HERE = next(p / "week06-parallelism" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "week06-parallelism" / "workers.py").exists())
sys.path.insert(0, str(HERE))          # so that "import workers" works, in this kernel and in child processes
import workers

DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
assert (DATA / "students.csv").exists(), "run  python data/make_data.py  from the repository root first"
print("Python", sys.version.split()[0], "| start method:", multiprocessing.get_start_method(),
      "| CPU cores:", multiprocessing.cpu_count())

## 1 · What is multiprocessing?

**Multiprocessing** is a system's ability to run more than one process at the same time: an application is divided
into smaller, independent routines, and the operating system places them on different CPU **cores**. A modern laptop
has 4, 8 or 16 cores; your Codespace has 2. A plain Python script uses one of them.

Why data scientists care (slide 4):

* **faster data loading**, by reading many files or API responses at once;
* **near-linear speed-up** for CPU-bound work (maths, feature engineering, model training) as you add cores;
* **larger datasets** processed in the same time.

Typical uses: loading many CSV/JSON files, engineering features on large tables, training several models in a
hyper-parameter search, scraping many pages, and preprocessing images.

The first thing to check is how many cores you have (slide 7). Every later decision about "how many workers"
starts from this number.

In [ ]:
import multiprocessing
print(multiprocessing.cpu_count())

> **Side note: counting usable cores.** The slide says `multiprocessing.cpu_count()`. That counts every core in the
> machine; a program may be allowed to use fewer (in containers and on shared servers). Current Python (3.13+) has
> `os.process_cpu_count()`, which counts the cores this process can actually use, and pools now use it as their
> default. The next cell works on every version. Read more: [What's new in Python 3.13](https://docs.python.org/whatsnew/3.13.html).

In [ ]:
import os
n_cores = getattr(os, "process_cpu_count", os.cpu_count)()   # usable cores; process_cpu_count is new in 3.13
print(n_cores)

The `multiprocessing` module has four kinds of building blocks (slide 7). This notebook uses all of them.

| Component | What it does | Section |
|---|---|---|
| `Process` | one child process running one function, with its own interpreter and memory | 5 |
| `Pool` | a team of worker processes that share out many tasks | 6–7 |
| `Queue` / `Pipe` | pass data safely between processes (inter-process communication, IPC) | 5, 9 |
| `Lock` / `Value` | coordinate access to a shared resource | 9 |

## 2 · Concurrency vs parallelism

The slides use a kitchen. **Concurrency** is one chef juggling several dishes: while the soup simmers, she chops the
salad. The dishes *overlap in time*, but at any instant she does one thing. **Parallelism** is several chefs, each
cooking a dish at the same moment.

![Threads are cooks sharing one kitchen and one knife (the GIL); processes are separate kitchens, each with its own cook, pantry and knife, connected by a hatch (Queue).](figures/kitchen.svg)

| | Concurrency | Parallelism |
|---|---|---|
| Meaning | tasks overlap in time; one core switches between them | tasks run at the same instant on different cores |
| Needs | one core is enough | several cores |
| Best for | **I/O-bound** work: waiting for disk, network, databases | **CPU-bound** work: maths, ML, transforms |
| Python module | `threading` | `multiprocessing` |
| Kitchen | one chef juggling several dishes | several chefs cooking at once |

A **process** is a running program with its own memory (its own kitchen). A **thread** is a line of execution
*inside* a process; all threads of a process share its memory (cooks in the same kitchen). Here are two threads from
the course's threading notebook. Each spends almost all its time waiting (`sleep`), which is exactly the I/O-bound
situation.

In [ ]:
import threading
import time

def print_numbers():
    for i in range(5):
        time.sleep(0.5)          # the course notebook uses 2 s; shortened here
        print(f"Number: {i}")

def print_letter():
    for letter in "abcde":
        time.sleep(0.5)
        print(f"Letter: {letter}")

t1 = threading.Thread(target=print_numbers)    # create the threads
t2 = threading.Thread(target=print_letter)

t = time.time()
t1.start()                                     # start them
t2.start()
t1.join()                                      # wait for both to finish
t2.join()
print(f"finished in {time.time() - t:.1f} s   (one after the other would take 5.0 s)")

The output interleaves, and the total is about 2.5 s instead of 5 s: while one thread sleeps, the other runs.
Notice that threads need no special set-up: functions defined in the notebook work as they are, because a thread
lives in the same process and sees the same memory.

## 3 · Python's Global Interpreter Lock (GIL)

The **GIL** is a mutex (a lock) inside the standard Python interpreter that lets only **one thread execute Python
bytecode at a time**. In the kitchen picture, the cooks share one knife. Waiting for a disk or a network does not need
the knife, so I/O-bound threads overlap well; but CPU-bound threads just take turns. Processes do not have this
problem: each process has its own interpreter and its own GIL, which is how multiprocessing **bypasses** it.

![Timelines: two CPU-bound threads take turns holding the GIL; two I/O-bound threads overlap their waiting; two processes run at the same time on two cores.](figures/gil.svg)

| Feature | `threading` | `multiprocessing` |
|---|---|---|
| True parallelism | no, in the standard build (the GIL prevents it) | yes (separate processes) |
| Memory space | shared | separate (isolated) |
| Best for | I/O-bound tasks | CPU-bound tasks |
| Overhead | low | higher (creating a process) |
| Data sharing | easy (shared memory) | needs IPC (`Queue`, `Pipe`) |

Do not take the table on trust: measure it. `workers.cpu_task` is pure arithmetic (CPU-bound) and
`workers.fake_download` only waits (I/O-bound).

> **Predict first.** On a 2-core machine, roughly how long will two CPU-bound tasks take with 2 threads, and with 2
> processes, compared with running them one after the other? And two I/O-bound tasks with 2 threads?

In [ ]:
N = 10_000_000

def run_threads(func, args_list):
    threads = [threading.Thread(target=func, args=(a,)) for a in args_list]
    for t in threads: t.start()
    for t in threads: t.join()

t0 = time.perf_counter(); [workers.cpu_task(N) for _ in range(2)];  seq = time.perf_counter() - t0
t0 = time.perf_counter(); run_threads(workers.cpu_task, [N, N]);     thr = time.perf_counter() - t0
t0 = time.perf_counter()
with multiprocessing.Pool(processes=2) as pool:
    pool.map(workers.cpu_task, [N, N])
prc = time.perf_counter() - t0
print(f"CPU-bound, 2 tasks:  one after the other {seq:.2f} s | 2 threads {thr:.2f} s | 2 processes {prc:.2f} s")

urls = [f"https://example.org/page{i}" for i in range(4)]
t0 = time.perf_counter(); [workers.fake_download(u) for u in urls]; seq = time.perf_counter() - t0
t0 = time.perf_counter(); run_threads(workers.fake_download, urls);  thr = time.perf_counter() - t0
print(f"I/O-bound, 4 tasks:  one after the other {seq:.2f} s | 4 threads {thr:.2f} s")

Typical result on a 2-core Codespace: the two threads are **no faster** than one after the other for the CPU-bound
work (sometimes slower), the two processes take clearly less time (a little over half, because starting the pool is
included in the timing), and the four I/O-bound threads finish in about the time of one download. Your numbers will
differ; the pattern will not.

> **Beyond the slides: when threads do help with computation.** The GIL limits only *Python bytecode*. NumPy, the C
> parts of pandas (such as the CSV parser), compression libraries and all I/O release it while they work, so threads
> can speed up work whose heavy part happens inside such libraries, without copying data to other processes. Measure
> before you choose. Read more: [glossary: global interpreter lock](https://docs.python.org/3/glossary.html#term-global-interpreter-lock).

> **Beyond the slides: Python without the GIL.** Python 3.13 added an optional *free-threaded* build without the GIL
> (`python3.13t`), and since Python 3.14 it is officially supported. On it, the CPU-bound threads above really do run
> in parallel. An extension that has not been updated still switches the GIL back on when it is imported, with a
> warning, and `sys._is_gil_enabled()` tells you which mode you are in. This course uses the standard build. Read more:
> [Python support for free threading](https://docs.python.org/3/howto/free-threading-python.html) and
> [What's new in Python 3.14](https://docs.python.org/whatsnew/3.14.html).

> **Rule of thumb (slide 19, in full in Section 11.1).** Waiting for disk, network or a database (I/O-bound) →
> threads. Heavy Python computation (CPU-bound) → processes. Many small independent tasks → a pool with `map`.

## 4 · Before you start a process: start methods, pickling and the `__main__` guard

A thread lives inside your process. A **child process** is a separate program, so Python must somehow give it the
function to run and the arguments. How it does that depends on the **start method**:

| Start method | How the child begins | Default on |
|---|---|---|
| `fork` | a copy of the parent, including every function you defined in the notebook | Linux, up to Python 3.13 |
| `spawn` | a fresh Python that **imports** the function by name | Windows, macOS |
| `forkserver` | fresh children made from a small server process; functions also imported by name | Linux, from Python 3.14 |

With `spawn` and `forkserver`, arguments and the function itself are sent to the child by **pickling**, and a
function is pickled only as its *name*, such as `__main__.square`. A notebook's functions exist only inside the
notebook, so the child cannot find them.

The next cell shows this on any operating system, by asking explicitly for `spawn`.

With **every** start method, `fork` included, a pool also pickles each *task* to send it to a worker. A `lambda`, a
function defined inside another function, an open file or a lock cannot be pickled, so `pool.map(lambda x: x * 2,
data)` fails even on Linux. Use a named function from `workers.py`. The second cell below shows the error.

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool

def square(x):                       # defined in the notebook
    return x ** 2

spawn = multiprocessing.get_context("spawn")

try:
    with ProcessPoolExecutor(max_workers=2, mp_context=spawn) as ex:
        print(list(ex.map(square, [1, 2, 3])))
except BrokenProcessPool as err:
    print("notebook function under spawn ->", type(err).__name__)

with ProcessPoolExecutor(max_workers=2, mp_context=spawn) as ex:
    print("workers.square under spawn   ->", list(ex.map(workers.square, [1, 2, 3])))

In [ ]:
with multiprocessing.Pool(processes=2) as pool:
    try:
        pool.map(lambda x: x * 2, [1, 2])
    except Exception as err:                       # PicklingError (AttributeError on some versions)
        print(type(err).__name__, "->", str(err)[:70])

The red text above is the children's own error report: `AttributeError: Can't get attribute 'square' on <module
'__main__'>`. The same function, imported from `workers.py`, works.

> **Look out: notebook functions and child processes.** In this Codespace (Linux, Python 3.12, `fork`), a function
> defined in a notebook *does* work in a child process. On Windows and macOS (`spawn`), and on Linux from
> Python 3.14 (`forkserver`), it does not: a `ProcessPoolExecutor` fails with `BrokenProcessPool`, and a
> `multiprocessing.Pool` is worse, because it **hangs** forever. The robust habit, used in every process example
> below: put the functions that child processes run in a `.py` file (here [`workers.py`](workers.py)) and import them.
> Threads do not have this problem. If a cell with a pool never finishes, interrupt it and **restart the kernel**; a
> `ProcessPoolExecutor` fails with `BrokenProcessPool` instead of hanging, which is one more reason to prefer it.
>
> Two more `fork` traps. `fork` copies the parent at the moment the pool is created, so a function you define *after*
> creating the pool does not exist in the workers, and the pool waits forever. And forking a program that runs threads,
> as every Jupyter kernel does, can deadlock the child, which is why Python 3.12 warns about it and Python 3.14 switched
> Linux to `forkserver`. Finally, after editing `workers.py`, restart the kernel (or run `importlib.reload(workers)`):
> otherwise, under `spawn` and `forkserver`, the notebook runs the old code while its workers import the new.
> Read more: [Contexts and start methods](https://docs.python.org/3/library/multiprocessing.html#contexts-and-start-methods).

> **Side note: the `__main__` guard.** The slides write process code under `if __name__ == "__main__":` and say it is
> "required on Windows to prevent recursive spawning". Current Python needs it wherever the start method is `spawn` or
> `forkserver`: Windows, macOS since Python 3.8, and Linux from Python 3.14. A child imports your **script** as a
> module; without the guard, that import runs the process-starting code again. In a notebook the guard is harmless
> (`__name__` is `"__main__"`), and in a `.py` script it is essential. Read more:
> [Programming guidelines](https://docs.python.org/3/library/multiprocessing.html#multiprocessing-programming).

## 5 · The `Process` class

### 5.1 · Define, create, start, join (slides 8–9)

Four steps: **define** a worker function, **create** `Process` objects with `target=` (the function) and `args=` (a
tuple of arguments), **start** them, and **join** them (wait for them to finish).

| Method | What it does | Must you call it? |
|---|---|---|
| `start()` | creates the child process and runs the target in it | yes |
| `join()` | blocks the parent until the child has finished | recommended |
| `is_alive()` | `True` while the child is still running | optional |

The slide's worker function, in `workers.py`:

```python
# workers.py
def count_numbers(nums):
    """Slide 9: print 0 .. nums-1, half a second apart."""
    for i in range(0, nums):
        print(i, "/", nums, flush=True)
        time.sleep(0.5)
```

In [ ]:
import multiprocessing
import time
from workers import count_numbers

if __name__ == "__main__":
    w1 = multiprocessing.Process(target=count_numbers, args=(5,))
    w2 = multiprocessing.Process(target=count_numbers, args=(6,))

    w1.start()  # Spawn process 1
    w2.start()  # Spawn process 2
    print("alive?", w1.is_alive(), w2.is_alive())
    w1.join()   # Wait for process 1
    w2.join()   # Wait for process 2
    print("alive?", w1.is_alive(), w2.is_alive(), "| exit codes:", w1.exitcode, w2.exitcode)
    print("Done!")

The two counts interleave because the two processes run at the same time; the whole cell takes about 3 s, the
length of the longer count, not 5.5 s. The lines may even come out jumbled (`1 /1 5 /`): both processes write to the
same screen at the same moment, and nothing coordinates them. An `exitcode` of 0 means the child finished normally.

`args=(5,)` needs the comma: `(5)` is just the number 5, and `args` must be a tuple.

> **Look out.** Output printed by child processes appears in the notebook here (Linux, `fork`). With `spawn` it may
> appear late, interleaved, or (on some systems) only in the terminal that runs Jupyter, so do not rely on `print`
> inside workers to get results back:
> **return** them (with a pool, Section 6) or send them through a `Queue` (below).

The course notebook's version uses two different functions, so you can see which process prints what:

In [ ]:
from workers import square_numbers, cube_numbers

if __name__ == "__main__":
    p1 = multiprocessing.Process(target=square_numbers)
    p2 = multiprocessing.Process(target=cube_numbers)
    t = time.time()
    p1.start(); p2.start()
    p1.join();  p2.join()
    print(f"finished in {time.time() - t:.1f} s")

### 5.2 · Each process has its own memory (slide 15)

A child process gets its **own copy** of the data. Changing a variable in the child does not change the parent's
variable: they are separate kitchens.

> **Predict first.** The child appends 99 to `values`. What does the parent print afterwards?

```python
# workers.py
def append_to_list(values, x):
    """Try to change a list in a child process (it changes only the child's copy)."""
    values.append(x)
    print("inside the child:", values, flush=True)
```

In [ ]:
values = [1, 2, 3]
p = multiprocessing.Process(target=workers.append_to_list, args=(values, 99))
p.start(); p.join()
print("in the parent:", values)

The parent's list is unchanged. To get data **back** from a child, send it through the hatch between the kitchens: a
`multiprocessing.Queue` (slide 7). The child calls `queue.put(item)`; the parent calls `queue.get()`, which waits
until an item arrives.

```python
# workers.py
def square_to_queue(n, queue):
    """Run in a child process: put (n, n squared) into a shared queue."""
    queue.put((n, n * n))
```

In [ ]:
q = multiprocessing.Queue()
procs = [multiprocessing.Process(target=workers.square_to_queue, args=(n, q)) for n in [2, 3, 4]]
for p in procs: p.start()
results = [q.get() for _ in procs]       # collect one item per process, in finishing order
for p in procs: p.join()
print(sorted(results))

> **Look out.** Take the items out of the queue **before** you `join()` the processes, as above. A child that has put
> a large item into a queue may not finish until someone reads it, so joining first can wait forever.

## 6 · Process pools

### 6.1 · The idea (slide 10)

Creating one `Process` per task is fine for two tasks, and wasteful for two hundred. A **pool** creates a fixed number
of worker processes once and hands them tasks one after another: when a worker finishes a task, it takes the next.

![Six tasks are submitted to a pool of two worker processes; each worker takes the next task when it is free, and the results are collected in order.](figures/pool.svg)

The lifecycle has four steps: **create** `Pool(processes=N)`, **submit** tasks with `apply()` or `apply_async()`,
**close** the pool with `close()` (no more tasks accepted), and **wait** with `join()`.

The slides' worker for this section returns a value:

```python
# workers.py
def count_total(nums):
    """Slides 11-12: print the running total and return it."""
    total = 0
    for i in range(0, nums):
        print(total, flush=True)
        total += i
        time.sleep(0.5)
    return total
```

### 6.2 · `Pool.apply()`: synchronous (slide 11)

In [ ]:
import multiprocessing
import time
from workers import count_total

if __name__ == "__main__":
    t = time.time()
    pool = multiprocessing.Pool(processes=2)
    result1 = pool.apply(count_total, args=(5,))
    result2 = pool.apply(count_total, args=(6,))
    pool.close()
    pool.join()
    print("Done!", result1, result2, f"in {time.time() - t:.1f} s")

`apply()` **blocks**: it waits for the result before your program continues, so task 2 starts only when task 1 has
finished. The pool has two workers, but only one is ever busy, and the total is about 5.5 s.

### 6.3 · `Pool.apply_async()`: asynchronous (slide 12)

In [ ]:
if __name__ == "__main__":
    t = time.time()
    pool = multiprocessing.Pool(processes=2)
    result1 = pool.apply_async(count_total, args=(5,))
    result2 = pool.apply_async(count_total, args=(6,))
    print("submitted; result1 is a", type(result1).__name__)
    pool.close()
    pool.join()
    print("Done!", result1.get(), result2.get(), f"in {time.time() - t:.1f} s")

`apply_async()` returns **immediately** with an `AsyncResult` object, a receipt for a result that does not exist yet,
so both tasks start at once on the two workers. `.get()` waits for the result and returns it. Total: about 3 s.

![Timelines: with apply, task 2 starts after task 1, about 5.5 s in total; with apply_async both run at once, about 3 s.](figures/apply.svg)

| Feature | `pool.apply()` | `pool.apply_async()` |
|---|---|---|
| Blocking? | yes: waits for the result | no: returns immediately |
| Return value | the result itself | an `AsyncResult`; call `.get()` |
| Parallelism | no: tasks run one after another | yes |
| Errors | raised at once | raised when you call `.get()` |
| Use when | each task needs the previous result | the tasks are independent |

The slides' best practice: **almost always prefer `apply_async()`**. `apply()` throws away the point of having a pool.

> **Side note: `close()` before `join()`.** The slides warn that `join()` "will hang forever" if `close()` has not
> been called. Current Python does not hang: `join()` on a pool that is still open raises
> `ValueError: Pool is still running`, as the next cell shows. The advice is the same: `close()`, then `join()`.
> Read more: [multiprocessing.pool.Pool](https://docs.python.org/3/library/multiprocessing.html#multiprocessing.pool.Pool).

In [ ]:
pool = multiprocessing.Pool(processes=2)
try:
    pool.join()
except ValueError as err:
    print("ValueError:", err)
pool.close()
pool.join()
print("closed, then joined: fine")

## 7 · The context manager, `map` and friends, and errors

### 7.1 · `with Pool(...) as pool:` (slides 16–17)

Forgetting `close()` and `join()` is easy. A `with` block cleans the pool up for you, even when an exception occurs:

In [ ]:
from workers import count_total_quiet          # count_total without the printing

if __name__ == "__main__":
    with multiprocessing.Pool(processes=2) as pool:
        r1 = pool.apply_async(count_total_quiet, args=(3,))
        r2 = pool.apply_async(count_total_quiet, args=(4,))
        print(r1.get(), r2.get())        # collect the results INSIDE the block
    # the pool is cleaned up here

> **Side note: what the `with` block really does.** The slides say the context manager gives "automatic close &
> join". Current Python's `Pool.__exit__` calls **`terminate()`**: it stops the workers at once, finished or not. The
> slides' example is safe only because it calls `.get()` inside the block. Ask for the result after the block and the
> task has been killed; the next cell waits 2 seconds and gives up. Rule: **collect every result inside the `with`
> block.** Read more: [multiprocessing.pool.Pool](https://docs.python.org/3/library/multiprocessing.html#multiprocessing.pool.Pool).

In [ ]:
with multiprocessing.Pool(processes=2) as pool:
    late = pool.apply_async(time.sleep, args=(1,))     # a 1-second task
# the block has ended: the pool was terminated before the task finished
try:
    late.get(timeout=2)
except multiprocessing.TimeoutError:
    print("TimeoutError: the result never arrived")

### 7.2 · Other useful pool methods (slide 18)

| Method | What it does | Returns |
|---|---|---|
| `map(func, iterable)` | parallel `map()`: `func` on every element; blocks until all are done | list of results |
| `map_async(func, iterable)` | non-blocking `map()` | `AsyncResult` |
| `starmap(func, iterable)` | like `map()`, but each element is a tuple of arguments | list of results |
| `imap(func, iterable)` | lazy `map()`: results come out one by one, in order | iterator |

In [ ]:
import multiprocessing
from workers import square, power, whoami

with multiprocessing.Pool(processes=4) as pool:      # the slide's value; on this 2-core Codespace, 2 is enough
    results = pool.map(square, [1, 2, 3, 4, 5, 6, 7, 8])
    print(results)

    pending = pool.map_async(square, range(5))
    print("map_async:", pending.get())

    print("starmap:", pool.starmap(power, [(2, 3), (3, 2), (10, 4)]))   # power(2, 3), power(3, 2), ...

    for value, worker in pool.imap(whoami, range(4)):
        print("imap:", value, "handled by", worker)

The results of `map` always come back **in the order of the input**, whichever worker finished first.

The slide's `square` is a one-liner, and in this Codespace you could define it right here in the notebook. The next
cell does that, but only when the start method is `fork`, so that it never hangs on Windows or macOS (Section 4):

In [ ]:
def square_here(x):
    return x ** 2

if multiprocessing.get_start_method() == "fork":
    with multiprocessing.Pool(processes=2) as pool:
        print("notebook function with fork:", pool.map(square_here, [1, 2, 3]))
else:
    print("start method is", multiprocessing.get_start_method(), "- use workers.square instead")

### 7.3 · Errors in workers (slide 16)

An exception inside a worker does not appear when it happens. It is stored, and **re-raised when you call `.get()`**.
Wrap `.get()` in `try`/`except`.

```python
# workers.py
def reciprocal(x):
    """1 / x; raises ZeroDivisionError for 0 (used to practise error handling)."""
    return 1 / x
```

In [ ]:
with multiprocessing.Pool(processes=2) as pool:
    tasks = [pool.apply_async(workers.reciprocal, args=(x,)) for x in [4, 0, 2]]
    for x, task in zip([4, 0, 2], tasks):
        try:
            print(x, "->", task.get())
        except ZeroDivisionError as err:
            print(x, "-> failed in the worker:", type(err).__name__, err)

### Mini challenge

Each call `count_total(n)` takes about `n × 0.5` seconds. With `Pool(processes=2)`, roughly how long does each version
take for the tasks 4, 4, 4, 4?

1. four `pool.apply(...)` calls;
2. four `pool.apply_async(...)` calls, then `.get()` on each;
3. `pool.map(count_total, [4, 4, 4, 4])`.

<details><summary>Answer</summary>

1. About 8 s: one task after another, 4 × 2 s.
2. About 4 s: two workers, so two rounds of two tasks.
3. About 4 s, for the same reason. `map` is the shortest way to write case 2 when every task uses the same function.

</details>

## 8 · `concurrent.futures`: one interface for threads and processes

The slides list `concurrent.futures` under further reading (slide 20), and the course's threading notebook uses it:
it gives threads and processes the **same** high-level interface. An *executor* is a pool; `submit()` is like
`apply_async()` and returns a `Future` (call `.result()`); `map()` works as in a `Pool`. It is the simplest tool for
most data-science jobs.

From the course notebook, a thread pool for I/O-bound work:

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def print_number(number):
    time.sleep(0.3)                       # the course notebook uses 1 s; shortened here
    return f"Number: {number}"

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3]

t = time.time()
with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(print_number, numbers)

for result in results:
    print(result)
print(f"{len(numbers)} tasks, 3 threads: {time.time() - t:.1f} s")

The course notebook's process-pool example. Only the class name changes, and the function comes from `workers.py`.
Note that `square_number` only *sleeps*, so it behaves like I/O: it shows the interface, not a CPU speed-up.

```python
# workers.py
def square_number(number):
    time.sleep(0.5)
    return f"Square: {number * number}"
```

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from workers import square_number

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 2, 3, 12, 14]
if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=3) as executor:
        results = executor.map(square_number, numbers)
        for result in results:
            print(result)

The course notebook's CPU-bound example computes large factorials in a pool. `factorial_digits` returns the number of
digits rather than the factorial itself, which would fill the screen:

```python
# workers.py
def factorial_digits(number):
    """How many decimal digits number! has."""
    sys.set_int_max_str_digits(0)          # allow converting very long integers to text
    return len(str(math.factorial(number)))
```

In [ ]:
numbers = [5000, 6000, 700, 8000]
t = time.time()
with ProcessPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(workers.factorial_digits, n) for n in numbers]   # submit = apply_async
    digits = [f.result() for f in futures]                                       # result = get
print(dict(zip(numbers, digits)), f"in {time.time() - t:.2f} s")

Each factorial takes only milliseconds, so here the pool's start-up costs more than it saves: profile first
(slide 16). Real CPU-bound work looks like this: four runs of the pure-Python arithmetic in `workers.cpu_task`,
one after the other and then on two processes.

In [ ]:
N = [5_000_000] * 4                       # workers.cpu_task: pure-Python arithmetic
t0 = time.perf_counter(); seq = [workers.cpu_task(n) for n in N]; s = time.perf_counter() - t0
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as ex:
    par = list(ex.map(workers.cpu_task, N))
p = time.perf_counter() - t0
print(f"CPU-bound x4: one after the other {s:.2f} s | 2 processes {p:.2f} s | same results: {seq == par}")

On the 2-core Codespace the two processes take a little over half the time. This is the case processes are for.

**When some tasks fail.** `future.result()` re-raises the worker's exception, like `.get()` in Section 7.3. Iterating
over `executor.map` stops at the **first** failing item and loses the rest, so when some tasks may fail (one bad URL
should not lose the other 99), use `submit` with `as_completed`, which hands you each future as soon as it finishes:

In [ ]:
from concurrent.futures import as_completed

with ProcessPoolExecutor(max_workers=2) as ex:
    futures = {ex.submit(workers.reciprocal, x): x for x in [4, 0, 2]}
    for fut in as_completed(futures):            # in finishing order, not input order
        x = futures[fut]
        try:
            print(x, "->", fut.result())         # re-raises the worker's exception here
        except ZeroDivisionError as err:
            print(x, "-> failed:", err)

> **Look out.** A plain `threading.Thread` whose function raises an exception only *prints* the traceback; `join()`
> returns normally and your program carries on as if nothing happened. Run the function through a
> `ThreadPoolExecutor` and call `.result()` to get the exception back. Read more:
> [threading.excepthook](https://docs.python.org/3/library/threading.html#threading.excepthook).

| `multiprocessing.Pool` | `concurrent.futures` executor |
|---|---|
| `Pool(processes=N)` | `ProcessPoolExecutor(max_workers=N)` or `ThreadPoolExecutor(max_workers=N)` |
| `pool.apply_async(f, args=(x,))` → `.get()` | `executor.submit(f, x)` → `.result()` |
| `pool.map(f, xs)` | `executor.map(f, xs)` |
| leaving `with` terminates the pool | leaving `with` **waits** for all submitted tasks |

Note the last row: an executor's `with` block waits for its tasks, which avoids the trap in Section 7.1.
Read more: [concurrent.futures](https://docs.python.org/3/library/concurrent.futures.html).

## 9 · Threads that share data: `Lock` and `Queue`

Threads share memory, which makes passing data easy (slide 6) and also dangerous. Slide 7 names the two tools that
make sharing safe: a **Lock** and a **Queue**.

### 9.1 · A race condition, and a `Lock`

Four cashiers update one account. Each deposit reads the balance, does something else for a moment, and writes back
balance + 1. If another thread writes in between, one of the two updates is lost. This is a **race condition**.

> **Predict first.** Four threads make 50 deposits each. What will the final balance be?

In [ ]:
import threading, time

def deposit_all(n_threads, n_deposits, lock=None):
    account = {"balance": 0}
    def deposit():
        for _ in range(n_deposits):
            if lock:
                lock.acquire()
            current = account["balance"]        # read
            time.sleep(0.0001)                  # another thread may run here
            account["balance"] = current + 1    # write
            if lock:
                lock.release()
    threads = [threading.Thread(target=deposit) for _ in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return account["balance"]

print("without a lock:", deposit_all(4, 50), "(expected 200)")
print("with a lock   :", deposit_all(4, 50, lock=threading.Lock()))

Without the lock most deposits are lost; with it, the result is exact. A `Lock` lets only one thread at a time into
the protected lines (the **critical section**). The usual way to write it is a `with` block, which always releases
the lock, even after an error:

```python
lock = threading.Lock()
with lock:
    current = account["balance"]
    account["balance"] = current + 1
```

A `Lock` is not re-entrant: acquiring it again in the same thread (a nested `with lock:`) waits forever, a
**deadlock**; use `threading.RLock` if you need that. With two locks, always acquire them in the same order in every
thread. Read more: [Python thread locks](https://realpython.com/python-thread-lock/).

The GIL does **not** protect you here: it switches threads between bytecode instructions, and a read-then-write is
several instructions. `multiprocessing` has the same tools for processes (`multiprocessing.Lock`, and `Value` for a
shared number).

### 9.2 · A `Queue` of work for threads

A `queue.Queue` is a thread-safe line of tasks: **producers** `put()` items in, **consumers** `get()` them out, and
the queue does all the locking. A common data-science pattern is a few worker threads that take file names from a
queue and process them. Each worker stops when it receives `None`, a *sentinel*.

In [ ]:
import queue, threading
import pandas as pd

paths = workers.make_student_files(DATA, n_files=4)     # students1.csv ... students4.csv (Section 10)

tasks, results = queue.Queue(), queue.Queue()

def consumer():
    while True:
        path = tasks.get()
        if path is None:                      # sentinel: no more work
            break
        df = pd.read_csv(path)
        results.put((path.name, len(df), threading.current_thread().name))

workers_threads = [threading.Thread(target=consumer, name=f"reader-{i}") for i in range(2)]
for t in workers_threads: t.start()
for p in paths: tasks.put(p)                  # the producer
for _ in workers_threads: tasks.put(None)     # one sentinel per worker
for t in workers_threads: t.join()

while not results.empty():
    print(results.get())

`queue.Queue` is for threads; `multiprocessing.Queue` (Section 5.2) is the version for processes. Both have `put()`
and `get()`.

> **Beyond the slides: other ways to finish a queue.** A producer can wait until every item has been *processed*
> (not just taken) with `tasks.join()`, if each consumer calls `tasks.task_done()` after finishing an item. Python 3.13
> also added `tasks.shutdown()`: consumers then get a `queue.ShutDown` exception once the queue is empty, so no
> sentinels are needed. The sentinel pattern above works on every version, including this Codespace's 3.12.
> Read more: [queue](https://docs.python.org/3/library/queue.html). Read more: [threading](https://docs.python.org/3/library/threading.html) and
[queue](https://docs.python.org/3/library/queue.html).

## 10 · Practical: parallel data loading with Pandas (slide 14)

The syllabus task for this week: **parallelise a data loading task**. First, make some files to load: the course's
`students.csv` split into four parts, `students1.csv` to `students4.csv`.

In [ ]:
paths = workers.make_student_files(DATA, n_files=4)
for p in paths:
    print(p.relative_to(DATA.parent), sum(1 for _ in open(p)) - 1, "rows")

The slide's reader, in `workers.py`, reads a file one row at a time:

```python
# workers.py
def read_student(filename):
    """Slide 14 as written: read one row at a time and grow a DataFrame."""
    import pandas as pd      # imported here so that starting a worker process stays fast
    students = pd.DataFrame()
    df = pd.read_csv(filename, chunksize=1)
    for d in df:
        students = pd.concat([students, d], axis=0, ignore_index=True)
    return students
```

In [ ]:
import multiprocessing
import pandas as pd
from workers import read_student

folder = DATA / "week06"
if __name__ == "__main__":
    pool = multiprocessing.Pool(processes=2)
    df1 = pool.apply_async(read_student, args=(folder / "students1.csv",))
    df2 = pool.apply_async(read_student, args=(folder / "students2.csv",))
    pool.close()
    pool.join()
    students = pd.concat([df1.get(), df2.get()], axis=0, ignore_index=True)
    print(students.shape)
    print(students.head())

Two processes read two files at the same time, and `pd.concat` combines the results once both have finished.

> **Side note: reading row by row.** The slide reads with `chunksize=1` and grows the DataFrame with `pd.concat` once
> per row, and itself notes that this is slow. Current Pandas reads a whole file in one fast call, `pd.read_csv(path)`;
> for files too big for memory, use a large `chunksize` (such as 100 000) and collect the chunks in a list, then call
> `pd.concat` **once**. Read more: [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

The modern version, for any number of files, is one `map` call:

```python
# workers.py
def read_csv_file(path):
    """The modern version: one call reads the whole file."""
    import pandas as pd
    return pd.read_csv(path)
```

In [ ]:
def load_sequential(paths):
    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

def load_with_processes(paths):
    with multiprocessing.Pool(processes=2) as pool:
        return pd.concat(pool.map(workers.read_csv_file, paths), ignore_index=True)

def load_with_threads(paths):
    with ThreadPoolExecutor(max_workers=4) as ex:
        return pd.concat(list(ex.map(pd.read_csv, paths)), ignore_index=True)

for f in (load_sequential, load_with_processes, load_with_threads):
    t0 = time.perf_counter()
    df = f(paths)
    print(f"{f.__name__:20s} {df.shape}  {time.perf_counter() - t0:.3f} s")

pd.testing.assert_frame_equal(load_sequential(paths), load_with_processes(paths))
print("all three give the same table")

On these small files, the plain sequential loop is usually **fastest**: starting processes and copying DataFrames
back from them costs more than reading 50 rows. With `spawn` or `forkserver` (Windows, macOS, Python 3.14) each
worker also imports pandas, about 0.3 s, so the break-even point is higher still. That is the slides' last best practice: **profile before you
parallelise**. Parallel loading pays off when there are many files, the files are large, or each read waits on a slow
disk or network. For network downloads (Project Stage 2), threads are the natural choice; for heavy per-file processing
after the read, processes are.

## 11 · Choosing, pitfalls and best practices

### 11.1 · Decision guide (slide 19)

| Situation | Examples | Use |
|---|---|---|
| CPU-bound task | maths, ML training, data transforms, image processing | `multiprocessing` / `ProcessPoolExecutor` |
| I/O-bound task | web requests, file I/O, database queries, API calls | `threading` / `ThreadPoolExecutor` (or `asyncio`) |
| Many independent tasks | 100 files, 1000 URLs, batch predictions | `Pool.map()` / `map_async()` / `executor.map()` |
| A few heavy tasks | train 2–3 models, process 2–3 large files | `Pool.apply_async()` / `executor.submit()` |

### 11.2 · Common pitfalls (slide 15)

| Severity | Pitfall | Fix |
|---|---|---|
| critical | no `if __name__ == "__main__":` guard in a script | always guard process-starting code (Section 4) |
| critical | worker functions defined only in a notebook | put them in a `.py` module (Section 4) |
| high | a `lambda` or nested function in `pool.map` | a named function in a `.py` module (Section 4) |
| high | `apply()` instead of `apply_async()` | `apply_async()` or `map()` (Section 6) |
| high | `join()` before `close()` | `close()` then `join()`, or a `with` block (Section 6) |
| high | reading results after a `with Pool` block | collect every `.get()` inside the block (Section 7) |
| medium | many more processes than cores | `processes=multiprocessing.cpu_count()` or fewer |
| medium | changing a variable in a child and expecting the parent to see it | return results, or use a `Queue` / `Manager` (Section 5) |
| medium | threads updating shared data without a lock | `threading.Lock` (Section 9) |

### 11.3 · Best practices (slide 16)

* **Use a pool for most tasks.** Manual `Process` objects are for long-running services.
* **Match the number of processes to the cores**, or leave `processes` out. Left out, `Pool()` and
  `ProcessPoolExecutor()` start one worker per usable core; `ThreadPoolExecutor()` starts `min(32, cores + 4)` threads,
  because I/O-bound threads mostly wait.
* **Keep worker functions simple** and their arguments picklable: numbers, strings, lists, DataFrames; not open files
  or database connections.
* **Use a context manager** (`with Pool(...) as pool:`) for automatic clean-up.
* **Profile before parallelising**: for small jobs, the overhead outweighs the gain (Section 10).
* **Handle exceptions** around `.get()` / `.result()` (Section 7.3).

### 11.4 · Further reading (slide 20)

* [**Celery**](https://docs.celeryq.dev/): a distributed task queue for running tasks on many machines, with
  scheduling, retries and monitoring.
* [**Apache Kafka**](https://kafka.apache.org/): a platform for streaming events and real-time data pipelines at very
  large scale.
* [**concurrent.futures**](https://docs.python.org/3/library/concurrent.futures.html): the unified thread/process pool
  interface of Section 8.
* [**Dask**](https://www.dask.org/): parallel NumPy- and Pandas-style computing on data larger than memory, with very
  few code changes.

## Summary (slide 21)

* **Concurrency** overlaps tasks in time (threads, best for I/O-bound work); **parallelism** runs them at the same
  instant on several cores (processes, best for CPU-bound work).
* The **GIL** lets one thread run Python code at a time; multiprocessing bypasses it with separate processes.
* The `Process` class gives fine control over single child processes: `start()`, `join()`, `is_alive()`. Each process
  has its own memory; send data back through a `Queue`.
* A `Pool` manages workers for you; prefer `apply_async()` or `map()` to `apply()`, and use `with Pool(...)`, collecting
  results inside the block. `concurrent.futures` offers the same ideas for threads and processes.
* Put process worker functions in a `.py` module and keep the `__main__` guard in scripts.
* Protect shared data in threads with a `Lock`; hand out work with a `Queue`.
* **Profile first**: parallelism adds overhead that pays off only for large or slow tasks.

**Project Stage 2** (due Week 9) asks for multithreaded or parallel data fetching or processing: the
`ThreadPoolExecutor` and `Pool.map` patterns of Sections 8 and 10 are the templates to start from.

Next: `02_exercises.ipynb`. Next topic: JSON file manipulation (Week 7).